In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv('malicious_phish.csv')

# Display the first few rows
print(df.head())

# Check for missing values
print(df.isnull().sum())

# Examine the distribution of labels
print(df['type'].value_counts())


                                                 url        type
0                                   br-icloud.com.br    phishing
1                mp3raid.com/music/krizz_kaliko.html      benign
2                    bopsecrets.org/rexroth/cr/1.htm      benign
3  http://www.garage-pirenne.be/index.php?option=...  defacement
4  http://adventure-nicaragua.net/index.php?optio...  defacement
url     0
type    0
dtype: int64
type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64


In [3]:
import re
import tldextract
from urllib.parse import urlparse

def extract_features(url):
    features = {}
    parsed = urlparse(url)
    ext = tldextract.extract(url)
    
    features['url_length'] = len(url)
    features['num_dots'] = url.count('.')
    features['num_hyphens'] = url.count('-')
    features['num_underscores'] = url.count('_')
    features['num_slashes'] = url.count('/')
    features['num_percent'] = url.count('%')
    features['num_question_marks'] = url.count('?')
    features['num_equals'] = url.count('=')
    features['num_at'] = url.count('@')
    features['has_ip'] = int(bool(re.search(r'\d+\.\d+\.\d+\.\d+', parsed.netloc)))
    features['has_https'] = int(parsed.scheme == 'https')
    features['domain_length'] = len(ext.domain)
    features['num_subdomains'] = len(ext.subdomain.split('.')) if ext.subdomain else 0
    features['path_length'] = len(parsed.path)
    
    return features

# Apply feature extraction to the dataset
feature_list = df['url'].apply(extract_features)
features_df = pd.DataFrame(feature_list.tolist())

# Combine the features with the original labels
df_features = pd.concat([features_df, df['type']], axis=1)


In [4]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_features['label'] = le.fit_transform(df_features['type'])
df_features.drop('type', axis=1, inplace=True)

In [5]:
print(df_features.head())

   url_length  num_dots  num_hyphens  num_underscores  num_slashes  \
0          16         2            1                0            0   
1          35         2            0                1            2   
2          31         2            0                0            3   
3          88         3            1                2            3   
4         235         2            1                1            3   

   num_percent  num_question_marks  num_equals  num_at  has_ip  has_https  \
0            0                   0           0       0       0          0   
1            0                   0           0       0       0          0   
2            0                   0           0       0       0          0   
3            0                   1           4       0       0          0   
4            0                   1           3       0       0          0   

   domain_length  num_subdomains  path_length  label  
0              9               0           16      3  
1     

In [6]:
df_features.to_csv('lexical_features.csv', index=False)